# Metodo di Newton: dal caso scalare ai sistemi non lineari

Questo notebook ricostruisce il metodo di Newton per la ricerca di zeri, prima nel caso scalare e poi per sistemi non lineari, mettendone in luce sia il comportamento ideale (convergenza quadratica) sia i modi in cui può fallire o degradare. Il percorso è organizzato in cinque step:

1. **Step 1 — Newton scalare e radice multipla.** Convergenza quadratica per radici semplici, convergenza lineare (molto più lenta) per radici di molteplicità maggiore di uno.
2. **Step 2 — Newton per sistemi: implementazione e caso regolare.** Generalizzazione a $\mathbb{R}^n$ tramite la matrice Jacobiana, verifica della convergenza quadratica su un sistema ben condizionato.
3. **Step 3 — Sensibilità al punto iniziale e soluzioni multiple.** Bacini di attrazione: punti di partenza diversi possono condurre a soluzioni diverse dello stesso sistema.
4. **Step 4 — Caso patologico: Jacobiano singolare.** Cosa succede, e perché, quando si parte esattamente in un punto dove la matrice Jacobiana non è invertibile.
5. **Step 5 — Condizionamento e sensibilità numerica.** La zona intermedia tra il caso regolare e quello singolare: Jacobiano invertibile ma mal condizionato.

## Perché il metodo di Newton

Il metodo prende il nome da Isaac Newton, che nel 1669 lo descrisse (in forma embrionale, applicata a un'equazione polinomiale specifica) come tecnica per approssimare successivamente le radici di un'equazione; Joseph Raphson ne pubblicò nel 1690 una formulazione più generale e vicina a quella moderna — da cui il nome alternativo "metodo di Newton-Raphson".

La sua importanza pratica viene dal fatto che la maggior parte delle equazioni (e dei sistemi di equazioni) non lineari che si incontrano in ingegneria, fisica e nelle scienze applicate non ammette una formula risolutiva chiusa: bisogna approssimarne le soluzioni numericamente. Newton è, tra i metodi iterativi per la ricerca di zeri, quello con la convergenza locale più rapida (quadratica), il che lo rende lo strumento di base su cui si costruiscono molti algoritmi più sofisticati — inclusi, con qualche accorgimento aggiuntivo, i metodi di ottimizzazione numerica. Ha però un prezzo: richiede di calcolare (o approssimare) le derivate, e la sua convergenza rapida è garantita solo localmente, non da un punto di partenza arbitrario — è proprio questo compromesso, insieme ai modi in cui può fallire, il filo conduttore di questo notebook.

## Step 1 — Newton scalare e radice multipla

### Il metodo di Newton per un'equazione scalare

Data un'equazione $f(x)=0$ con $f$ derivabile, il metodo di Newton costruisce una successione di approssimazioni $x_k$ linearizzando $f$ attorno all'iterata corrente. Sviluppando $f$ in serie di Taylor al prim'ordine attorno a $x_k$,

$$f(x) \approx f(x_k) + f'(x_k)(x-x_k),$$

e imponendo che l'approssimazione lineare si annulli in $x_{k+1}$, si ottiene l'iterazione

$$x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}.$$

Geometricamente, $x_{k+1}$ è l'intersezione con l'asse delle ascisse della retta tangente al grafico di $f$ nel punto $(x_k, f(x_k))$.

### Ordine di convergenza e radici multiple

Se $x^*$ è una radice **semplice** di $f$ (cioè $f(x^*)=0$ e $f'(x^*)\neq 0$) e $f$ è sufficientemente regolare, l'errore $e_k = x_k - x^*$ soddisfa asintoticamente

$$e_{k+1} \approx \frac{f''(x^*)}{2f'(x^*)}\,e_k^2,$$

cioè il numero di cifre corrette raddoppia a ogni iterazione: convergenza **quadratica**.

Se invece $x^*$ è una radice di **molteplicità algebrica** $m>1$ (cioè $f(x^*)=f'(x^*)=\dots=f^{(m-1)}(x^*)=0$ ma $f^{(m)}(x^*)\neq 0$), lo sviluppo cambia: vicino a $x^*$, $f(x)\approx c\,(x-x^*)^m$ con $c\neq 0$, e si può mostrare che

$$e_{k+1} \approx \frac{m-1}{m}\,e_k,$$

cioè l'errore si riduce solo di un fattore costante $\frac{m-1}{m}<1$ a ogni passo: la convergenza diventa **lineare**, molto più lenta di quella quadratica. Nella pratica questo si osserva come un numero di iterazioni necessarie a raggiungere una data tolleranza sensibilmente più alto rispetto al caso di radice semplice, a parità di punto di partenza e di tolleranza richiesta.

### Import e librerie

Importiamo `numpy` per il calcolo numerico, `matplotlib.pyplot` per i grafici e `sympy` per il calcolo simbolico. Le due funzioni riutilizzabili di questo notebook, `newton` e `newton_vett`, e l'helper `to_numeric` (che converte un'espressione simbolica in una funzione numerica) non sono definite qui: vivono nei moduli condivisi `src/newton.py` e `src/symbolic_utils.py`.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

from src.symbolic_utils import to_numeric
from src.newton import newton, newton_vett

### La funzione di prova: una radice doppia in $x=2$

Come esempio si usa

$$f(x) = (x-2)^2\, e^{x}.$$

Il fattore $e^x$ non si annulla mai, quindi l'unico zero di $f$ è $x^*=2$, generato interamente dal fattore $(x-2)^2$: si tratta perciò di una radice di molteplicità algebrica esattamente $m=2$ (una radice doppia "pulita", non mescolata con zeri di altri fattori).

In [ ]:
x = sp.symbols('x')
fsym = (x - 2)**2 * sp.exp(x)
dfsym = sp.diff(fsym, x)
print('f(x)  =', fsym)
print("f'(x) =", dfsym)

f = to_numeric(fsym, x)
df = to_numeric(dfsym, x)

### Implementazione dell'algoritmo

L'iterazione di Newton vista sopra viene implementata come funzione `newton(f, df, x0, atol, rtol, nmax)`. Il criterio d'arresto è sull'incremento relativo tra due iterate successive: ci si ferma quando

$$\frac{|x_{k+1}-x_k|}{1+\frac{\mathrm{rtol}}{\mathrm{atol}}|x_k|} \le \mathrm{atol},$$

oppure dopo `nmax` iterazioni.

Se si parte da un punto $x_0$ lontano dalla radice doppia $x^*=2$, ci si aspetta — per quanto detto sopra sulla molteplicità — che il numero di iterazioni necessario per raggiungere la tolleranza sia molto maggiore di quello tipico di una radice semplice (che con le stesse tolleranze richiederebbe tipicamente meno di una decina di iterazioni).

Eseguiamo `newton` a partire da $x_0=8$, lontano dalla radice doppia $x^*=2$.

In [ ]:
x0 = 8.0
xs, fs, n_it, all_x = newton(f, df, x0, atol=1e-8, rtol=1e-8, nmax=100)
print('soluzione xs =', xs)
print('f(xs) =', fs)
print('numero iterate =', n_it)

### Interpretazione: la firma della radice doppia

Partendo da $x_0=8$, a una distanza di 6 dalla radice, il metodo impiega **35 iterazioni** per soddisfare la tolleranza $\mathrm{atol}=\mathrm{rtol}=10^{-8}$. È un numero elevato: per una radice semplice, con lo stesso punto di partenza e le stesse tolleranze, la convergenza quadratica farebbe raddoppiare le cifre corrette a ogni passo, portando tipicamente a convergenza in meno di una decina di iterazioni.

Qui $m=2$, quindi asintoticamente $e_{k+1}\approx \tfrac{1}{2}e_k$, cioè **un solo bit di precisione guadagnato per iterazione** invece che un raddoppio delle cifre corrette. Per portare l'errore iniziale (dell'ordine di $10^0$–$10^1$) fino a $10^{-8}$ servono quindi dell'ordine di $\log_2(10^{8})\approx 27$ dimezzamenti — in linea con le 35 iterazioni osservate (a cui si sommano le prime iterazioni, ancora lontane dal regime asintotico). La convergenza lineare della radice doppia è così visibile direttamente nel numero di iterazioni, senza bisogno di altro.

## Step 2 — Newton per sistemi: implementazione e caso regolare

### Generalizzazione a più dimensioni

Per un sistema di $n$ equazioni non lineari in $n$ incognite, scritto come $F(x)=0$ con $F:\mathbb{R}^n\to\mathbb{R}^n$, lo stesso argomento si applica linearizzando $F$ con lo sviluppo di Taylor al prim'ordine, ora vettoriale:

$$F(x) \approx F(x_k) + J(x_k)(x-x_k), \qquad J(x_k) = \left[\frac{\partial F_i}{\partial x_j}(x_k)\right]_{i,j},$$

dove $J$ è la matrice Jacobiana di $F$. Imponendo che l'approssimazione lineare si annulli in $x_{k+1}$ si ottiene, invece di una divisione come nel caso scalare, un **sistema lineare** da risolvere a ogni iterazione:

$$J(x_k)\,\Delta x = F(x_k), \qquad x_{k+1} = x_k - \Delta x.$$

Se $x^*$ è una soluzione con $J(x^*)$ invertibile (l'analogo vettoriale di $f'(x^*)\neq 0$) e $F$ è sufficientemente regolare, vale ancora convergenza **quadratica**: in norma, $\|e_{k+1}\| \approx C\,\|e_k\|^2$ per una costante $C$ che dipende dalle derivate seconde di $F$ in $x^*$ — il numero di cifre corrette raddoppia a ogni passo, esattamente come nel caso scalare con radice semplice.

### Il sistema di prova: circonferenza e iperbole

Come primo esempio, ben condizionato, si usa l'intersezione di una circonferenza e un'iperbole equilatera:

$$f_1(x,y) = x^2+y^2-4 = 0, \qquad f_2(x,y) = xy-1 = 0,$$

La soluzione: dalla seconda equazione $y=1/x$ (con $x\neq0$), sostituendo nella prima si ottiene

$$x^2+\frac{1}{x^2}=4 \;\Longrightarrow\; u^2-4u+1=0, \quad u=x^2,$$

che ha soluzioni $u = 2\pm\sqrt3$, entrambe positive. Ci sono quindi **quattro soluzioni reali**:

$$x=\pm\sqrt{2+\sqrt3},\; y=1/x \qquad\text{e}\qquad x=\pm\sqrt{2-\sqrt3},\; y=1/x,$$

usate più sotto per verificare il risultato numerico.

In [ ]:
def f_sys(v):
    x, y = v
    return np.array([x**2 + y**2 - 4, x*y - 1])

x_sym = sp.symbols('x0:2')
X = sp.Matrix(x_sym)
fsym_sys = sp.Matrix([X[0]**2 + X[1]**2 - 4, X[0]*X[1] - 1])
dfsym_sys = fsym_sys.jacobian(X)
print('J(x,y) =', dfsym_sys)

df_sys = to_numeric(dfsym_sys, [x_sym])

### Esecuzione da un punto iniziale ragionevole

Si parte da $x_0=(1.5,\,0.7)$, vicino alla soluzione analitica $\big(\sqrt{2+\sqrt3},\,1/\sqrt{2+\sqrt3}\big)\approx(1.932,\,0.518)$ ma non coincidente con essa. Essendo lo Jacobiano invertibile in un intorno di questa soluzione (il sistema è ben condizionato lì), ci si aspetta la convergenza quadratica descritta sopra: nello storico dell'errore normwise `all_err`, ogni valore dovrebbe essere circa il quadrato del precedente, cioè il numero di cifre corrette dovrebbe raddoppiare a ogni iterazione.

In [ ]:
x0_sys = np.array([1.5, 0.7])
xs_sys, fs_sys, dfs_sys, n_it_sys, all_err_sys = newton_vett(f_sys, df_sys, x0_sys, atol=1e-10, rtol=1e-10, nmax=100)

x_analytic = np.sqrt(2 + np.sqrt(3))
sol_analytic = np.array([x_analytic, 1/x_analytic])

print('soluzione xs =', xs_sys)
print('soluzione analitica =', sol_analytic)
print('f(xs) =', fs_sys)
print('numero iterate =', n_it_sys)
print('storico errore normwise =', all_err_sys)

### Interpretazione: convergenza quadratica confermata

Il metodo converge in **6 iterazioni** a $x_s\approx(1.93185165,\,0.51763809)$, che coincide (fino alla precisione stampata) con la soluzione analitica $\big(\sqrt{2+\sqrt3},\,1/\sqrt{2+\sqrt3}\big)$ derivata sopra, e $F(x_s)$ è nullo entro la precisione di macchina.

Lo storico dell'errore normwise è

$$0.237,\;\; 0.0501,\;\; 3.94\times10^{-3},\;\; 2.34\times10^{-5},\;\; 8.19\times10^{-10},\;\; 2.14\times10^{-17}.$$

Guardando il numero di cifre corrette, cioè $-\log_{10}(\text{errore})$, si ottiene approssimativamente $0.6,\,1.3,\,2.4,\,4.6,\,9.1$ (l'ultimo valore satura alla precisione di macchina): la sequenza **raddoppia** a ogni iterazione, la firma della convergenza quadratica $\|e_{k+1}\|\approx C\|e_k\|^2$ prevista dalla teoria.

## Step 3 — Sensibilità al punto iniziale e soluzioni multiple

### Bacini di attrazione

Il metodo di Newton è un metodo **locale**: la sua convergenza (quando c'è) è garantita solo in un intorno sufficientemente piccolo di una soluzione, non su tutto il dominio. Quando un sistema $F(x)=0$ ammette più soluzioni isolate $x_1^*, x_2^*,\dots$, ciascuna ha un proprio **bacino di attrazione**: l'insieme dei punti di partenza $x_0$ da cui la successione di Newton converge proprio a quella soluzione. Punti di partenza diversi, anche relativamente vicini tra loro, possono quindi cadere in bacini diversi e produrre soluzioni finali diverse — a differenza di un metodo globale, non c'è una regola semplice a priori che dica quale soluzione si troverà, se non l'intuizione geometrica di "vicinanza" al punto di partenza.

### Il secondo sistema: parabola e circonferenza

Si usa l'intersezione di una parabola e una circonferenza:

$$f_1(x,y) = x^2-y = 0, \qquad f_2(x,y) = x^2+y^2-1 = 0,$$

Sostituendo $y=x^2$ (dalla prima equazione) nella seconda si ottiene

$$x^4+x^2-1=0 \;\Longrightarrow\; u^2+u-1=0, \quad u=x^2,$$

la cui unica radice ammissibile (l'altra è negativa, e $u=x^2$ dev'essere $\ge0$) è $u=\dfrac{\sqrt5-1}{2}\approx0.618$. Si ottengono così **due soluzioni reali**, simmetriche rispetto all'asse $y$:

$$x=\pm\sqrt{u}\approx\pm0.786, \qquad y=u\approx0.618.$$

In [ ]:
def f_sys2(v):
    x, y = v
    return np.array([x**2 - y, x**2 + y**2 - 1])

fsym_sys2 = sp.Matrix([X[0]**2 - X[1], X[0]**2 + X[1]**2 - 1])
dfsym_sys2 = fsym_sys2.jacobian(X)
print('J(x,y) =', dfsym_sys2)

df_sys2 = to_numeric(dfsym_sys2, [x_sym])

u = (np.sqrt(5) - 1) / 2
sol_pos = np.array([np.sqrt(u), u])
sol_neg = np.array([-np.sqrt(u), u])
print('soluzione analitica +:', sol_pos)
print('soluzione analitica -:', sol_neg)

### Quattro punti di partenza

Si lancia `newton_vett` (già definita nello Step 2, riusata qui senza modifiche) da quattro punti iniziali diversi: due nel semipiano $x>0$, dove ci si aspetta convergenza verso la soluzione positiva, e due in $x<0$, verso quella negativa — per verificare che sia proprio il segno (la posizione) del punto di partenza a decidere l'esito, non un dettaglio implementativo.

In [ ]:
starting_points = [np.array([1.0, 1.0]), np.array([2.0, 2.0]),
                   np.array([-1.0, 1.0]), np.array([-2.0, 2.0])]

print(f"{'x0':<15}{'xs':<28}{'n_it':<6}")
for x0_i in starting_points:
    xs_i, fs_i, dfs_i, n_it_i, _ = newton_vett(f_sys2, df_sys2, x0_i, atol=1e-10, rtol=1e-10, nmax=100)
    x0_str = f"({x0_i[0]:.1f}, {x0_i[1]:.1f})"
    xs_str = f"({xs_i[0]:.8f}, {xs_i[1]:.8f})"
    print(f"{x0_str:<15}{xs_str:<28}{n_it_i:<6}")

### Interpretazione: il segno del punto di partenza decide la soluzione

I quattro punti di partenza convergono esattamente come anticipato:

| $x_0$ | soluzione trovata | iterazioni |
|---|---|---|
| $(1.0,\,1.0)$ | $(0.78615138,\,0.61803399)$ | 5 |
| $(2.0,\,2.0)$ | $(0.78615138,\,0.61803399)$ | 7 |
| $(-1.0,\,1.0)$ | $(-0.78615138,\,0.61803399)$ | 5 |
| $(-2.0,\,2.0)$ | $(-0.78615138,\,0.61803399)$ | 7 |

I due punti con $x_0>0$ cadono entrambi nel bacino di attrazione della soluzione positiva $(0.786,\,0.618)$, i due con $x_0<0$ in quello della soluzione negativa $(-0.786,\,0.618)$ — entrambe coincidenti, fino alla precisione stampata, con le soluzioni analitiche derivate sopra. Il segno di $x_0$ (la sua posizione rispetto all'asse di simmetria $x=0$ del sistema) determina quindi quale delle due soluzioni viene trovata, esattamente il comportamento previsto dalla teoria dei bacini di attrazione: **due punti di partenza diversi, due soluzioni diverse**, pur essendo il sistema e l'algoritmo identici. Si nota anche, di riflesso, che partire più lontano dalla soluzione (i punti $(\pm2,2)$ contro $(\pm1,1)$) costa qualche iterazione in più, pur restando nello stesso bacino.

## Step 4 — Jacobiano singolare

### Perché serve $J(x_k)$ invertibile

Il passo di Newton per sistemi, visto nello Step 2, richiede a ogni iterazione la soluzione del sistema lineare

$$J(x_k)\,\Delta x = F(x_k).$$

Questo è un sistema lineare $n\times n$ nell'incognita $\Delta x$: ha soluzione unica se e solo se $J(x_k)$ è invertibile, cioè $\det J(x_k)\neq0$. Se $\det J(x_k)=0$, il sistema non ha soluzione unica (può non averne affatto, o averne infinite), e non esiste un modo univoco di definire $x_{k+1}$: l'iterazione di Newton è, in quel punto, semplicemente non definita. Numericamente, `numpy.linalg.solve` rileva la singolarità (o quasi-singolarità) della matrice e solleva un'eccezione (`LinAlgError: Singular matrix`) invece di restituire un risultato arbitrario.

### Un punto in cui $J$ è singolare, individuato a mano

Per il sistema parabola/circonferenza dello Step 3, lo Jacobiano è

$$J(x,y) = \begin{pmatrix} 2x & -1 \\ 2x & 2y \end{pmatrix}, \qquad \det J(x,y) = 2x\cdot2y - (-1)\cdot2x = 4xy+2x = 2x(2y+1),$$

che si annulla per $x=0$ oppure $y=-\tfrac12$. Si sceglie il punto $(0,0)$: è il vertice della parabola $y=x^2$, ma soprattutto è il centro della circonferenza $x^2+y^2=1$, dove il gradiente di $f_2$ si annulla identicamente ($\partial f_2/\partial x = 2x=0$, $\partial f_2/\partial y=2y=0$). La seconda riga di $J(0,0)$ è dunque interamente nulla: non è un incidente numerico, ma una conseguenza diretta della geometria di $f_2$ in quel punto.

In [ ]:
det_sym = dfsym_sys2.det()
print('det J(x,y) =', det_sym)
print('det J(0,0) =', det_sym.subs({X[0]: 0, X[1]: 0}))

Verificato che $\det J(0,0)=0$, tentiamo comunque di lanciare `newton_vett` proprio da quel punto, per osservare l'errore che ne consegue.

In [ ]:
try:
    newton_vett(f_sys2, df_sys2, np.array([0.0, 0.0]), atol=1e-10, rtol=1e-10, nmax=100)
except np.linalg.LinAlgError as e:
    print(f'LinAlgError: {e}')

### Interpretazione: un errore atteso, non un incidente

Il calcolo simbolico conferma $\det J(0,0)=0$, e lanciando `newton_vett` proprio da $(0,0)$ si ottiene esattamente `LinAlgError: Singular matrix` alla primissima iterazione — l'errore è stato riprodotto intenzionalmente. La ragione è quella discussa sopra: il passo di Newton richiede di risolvere $J(x_k)\Delta x=F(x_k)$, e in $(0,0)$ la seconda riga di $J$ è nulla, quindi la seconda equazione del sistema linearizzato diventa $0=f_2(0,0)=-1$, priva di soluzione in $\Delta x$. `numpy.linalg.solve` non può quindi determinare univocamente il passo successivo e solleva l'eccezione invece di proseguire con un risultato arbitrario o numericamente instabile. Questo è il prezzo della genericità del metodo di Newton: senza un'ipotesi di invertibilità dello Jacobiano lungo tutta la traiettoria, l'algoritmo può semplicemente non essere definito in certi punti.

## Step 5 — Condizionamento e sensibilità numerica

### La zona grigia: mal condizionato, non singolare

Tra il caso regolare (Step 2) e quello esattamente singolare (Step 4) c'è una zona intermedia: punti in cui $J(x_k)$ è invertibile, ma **mal condizionata**. Il numero di condizionamento

$$\mathrm{cond}(J) = \frac{\sigma_{\max}(J)}{\sigma_{\min}(J)}$$

(rapporto tra il valore singolare massimo e quello minimo) misura quanto il passo $\Delta x = J(x_k)^{-1}F(x_k)$ amplifica le perturbazioni presenti in $F(x_k)$ — errori di arrotondamento in aritmetica floating point, in primo luogo. Se $\mathrm{cond}(J)\approx1$ le perturbazioni non vengono amplificate; se $\mathrm{cond}(J)$ è grande, anche un errore minuscolo su $F(x_k)$ può produrre un passo $\Delta x$ enormemente sbagliato. Poiché $J$ diventa esattamente singolare in $(0,0)$ (Step 4), ci si aspetta che $\mathrm{cond}(J(x,y))\to\infty$ man mano che $(x,y)\to(0,0)$.

### Un punto vicino, ma non uguale, al punto singolare

Si sceglie $x_0=(0.001,\,0.001)$: vicino a $(0,0)$ (il punto singolare dello Step 4), ma diverso — abbastanza vicino da restare nella zona di forte mal condizionamento, senza cadere nel caso esattamente singolare già trattato.

In [ ]:
x0_cond = np.array([0.001, 0.001])
cond_x0 = np.linalg.cond(df_sys2(x0_cond))
print('J(x0) =', df_sys2(x0_cond))
print('cond(J(x0)) =', cond_x0)

Eseguiamo `newton_vett` da $x_0$ e osserviamo l'andamento dell'errore in scala semilogaritmica.

In [ ]:
xs_cond, fs_cond, dfs_cond, n_it_cond, all_err_cond = newton_vett(f_sys2, df_sys2, x0_cond, atol=1e-10, rtol=1e-10, nmax=100)
print('soluzione xs =', xs_cond)
print('numero iterate =', n_it_cond)

plt.figure(figsize=(6, 4))
plt.semilogy(range(1, n_it_cond + 1), all_err_cond, 'o-')
plt.xlabel('iterazione $k$')
plt.ylabel('errore normwise relativo')
plt.title('Convergenza da un punto mal condizionato')
plt.grid(True, which='both', alpha=0.3)
plt.show()

### Interpretazione: il condizionamento elevato si vede nella traiettoria

Il numero di condizionamento calcolato è $\mathrm{cond}(J(x_0))\approx499$ — molto più alto del condizionamento di $J$ vicino alla soluzione regolare dello Step 2 (dell'ordine dell'unità). L'effetto si vede chiaramente nel grafico: il metodo impiega **15 iterazioni** invece delle 5-7 viste nello Step 3 per punti di partenza ben condizionati, e soprattutto la forma della curva è tutt'altro che il calo pulito e monotono dello Step 2. Si distinguono tre fasi:

1. **Salto iniziale enorme** (iterazione 1, errore $\approx5\times10^2$): il primo passo di Newton, calcolato con uno Jacobiano quasi singolare, viene fortemente amplificato — esattamente il comportamento previsto da $\mathrm{cond}(J)$ grande, che scaraventa l'iterata lontano dal punto di partenza.
2. **Plateau lento** (iterazioni 2-11, errore che scende molto gradualmente da $\approx0.8$ a $\approx0.14$): dopo il salto, l'iterata si trova in una zona dove il progresso è lento, ben lontano dal dimezzamento o dal raddoppio delle cifre corrette.
3. **Rientro nella convergenza quadratica** (iterazioni 12-15): una volta che l'iterata è abbastanza vicina alla soluzione $(0.786,\,0.618)$, dove $J$ torna ben condizionata, riemerge la stessa convergenza quadratica dello Step 2 — l'errore crolla da $\approx3\times10^{-2}$ a $\approx5\times10^{-12}$ in sole 3 iterazioni.

Il condizionamento elevato in $x_0$ non impedisce dunque la convergenza in questo caso, ma la rende inizialmente lenta e imprevedibile, in netto contrasto con la traiettoria pulita osservata quando si parte da una zona ben condizionata: è la manifestazione pratica, nel numero di iterazioni e nella forma della curva, della sensibilità numerica quantificata da $\mathrm{cond}(J)$.

## Conclusioni e limiti

I cinque step di questo notebook tracciano un percorso coerente: dalla convergenza ideale a quella che si rompe, passando per le sfumature intermedie.

- **Convergenza quadratica in condizioni regolari** (Step 2): con radice/soluzione semplice e Jacobiano invertibile, il numero di cifre corrette raddoppia a ogni iterazione — poche iterazioni bastano per raggiungere la precisione di macchina.
- **Convergenza lineare per radici multiple** (Step 1): quando la molteplicità algebrica è $m>1$, la convergenza quadratica si perde e si ottiene solo convergenza lineare con rapporto $(m-1)/m$ — servono molte più iterazioni per la stessa tolleranza.
- **Dipendenza dal punto di partenza** (Step 3): in presenza di più soluzioni, Newton converge a quella nel cui bacino di attrazione cade $x_0$ — non esiste una regola a priori (se non l'euristica della vicinanza) per prevedere quale.
- **Rottura in corrispondenza di uno Jacobiano singolare** (Step 4): il passo di Newton è definito solo dove $J(x_k)$ è invertibile; se non lo è, l'algoritmo semplicemente non può proseguire.
- **Sensibilità numerica per condizionamento elevato** (Step 5): anche restando nel regime "regolare" (Jacobiano invertibile), un condizionamento alto amplifica gli errori e produce una traiettoria di convergenza inizialmente lenta o irregolare, prima di rientrare nel regime quadratico vicino alla soluzione.

Un limite comune a tutti gli esperimenti: l'implementazione qui usata è Newton "puro", senza alcun meccanismo di correzione del passo (line search, backtracking, smorzamento). Questo lo rende semplice da analizzare e da collegare direttamente alla teoria, ma anche fragile: la sua convergenza è garantita solo localmente, e nulla nell'algoritmo stesso protegge da un punto di partenza sfortunato, da una soluzione multipla o da uno Jacobiano mal condizionato — la responsabilità di scegliere bene il punto iniziale, e di riconoscere quando l'algoritmo sta fallendo, resta interamente di chi lo usa.